# Postproceso Anatomico Thoracolumbar V2 Explicado - Colab

Este notebook implementa una segunda version del postproceso anatomico para las
predicciones thoracolumbares.

## Motivacion

La version anterior de postproceso redujo las etiquetas extra, pero fue demasiado
agresiva y destruyo casi toda la segmentacion util. Eso nos dio una conclusion
importante:

- el problema principal si es la sobreprediccion en imagenes parciales
- pero no conviene imponer un recorte anatomico duro

## Objetivo de esta version

Construir un postproceso **conservador** que:

1. limpie ruido pequeno
2. preserve la mayor parte de la mascara valida
3. reduzca clases aisladas improbables
4. respete el orden anatomico vertical
5. compare resultados antes y despues

## 0. Preparacion de Colab

Ajusta `PROJECT_ROOT` si en tu Google Drive el proyecto esta en otra ruta.

In [ ]:
import os
from pathlib import Path


PROJECT_ROOT_CANDIDATES = [
    Path.cwd() / "data" / "Scoliosis_Dataset",
    Path.cwd().parent / "data" / "Scoliosis_Dataset",
    Path.cwd().parent.parent / "data" / "Scoliosis_Dataset",
    Path.cwd() / "Scoliosis_Dataset",
    Path.cwd().parent / "Scoliosis_Dataset",
]

PROJECT_ROOT = next((path for path in PROJECT_ROOT_CANDIDATES if path.exists()), None)
if PROJECT_ROOT is None:
    searched = "\n".join(str(path) for path in PROJECT_ROOT_CANDIDATES)
    raise FileNotFoundError(
        "No se encontro data/Scoliosis_Dataset. Rutas revisadas:\n" + searched
    )

os.chdir(PROJECT_ROOT)
print('Working directory:', Path.cwd())

## 1. Librerias y configuracion general

Esta configuracion sigue la misma base del experimento `partial` que hasta ahora
ha sido el mas fuerte dentro de la linea exploratoria multiclase.

In [ ]:
from __future__ import annotations

import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage as ndi

import torch
import torch.nn as nn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

CWD = Path.cwd()

ROOT_CANDIDATES = [
    CWD,
    CWD / "data" / "Scoliosis_Dataset",
    CWD.parent / "data" / "Scoliosis_Dataset",
    CWD.parent.parent / "data" / "Scoliosis_Dataset",
    CWD / "Scoliosis_Dataset",
    CWD.parent / "Scoliosis_Dataset",
]

REQUIRED_FILES = [
    "indice_dataset.csv",
    "diccionario_etiquetas_T1_T12_L1_L5.json",
]


def is_valid_root(path: Path) -> bool:
    return path.exists() and all((path / rel).exists() for rel in REQUIRED_FILES)


ROOT = next((p for p in ROOT_CANDIDATES if is_valid_root(p)), None)
assert ROOT is not None, (
    "No se pudo localizar la carpeta Scoliosis_Dataset con los archivos esperados. "
    f"Directorio actual: {CWD}"
)

ROOT = ROOT.resolve()
PROJECT_DIR_CANDIDATES = [ROOT.parent.parent, *ROOT.parents, CWD, *CWD.parents]
PROJECT_DIR = next(
    (
        path
        for path in PROJECT_DIR_CANDIDATES
        if (path / "notebooks").exists() and (path / "data").exists()
    ),
    ROOT.parent.parent,
)
REPORTS_DIR = PROJECT_DIR / "reports"
ANALYSIS_DIR = REPORTS_DIR / "analysis_outputs"
OUTPUTS_DIR = PROJECT_DIR / "outputs"
MODELS_DIR = PROJECT_DIR / "models"
for directory in [REPORTS_DIR, ANALYSIS_DIR, OUTPUTS_DIR, MODELS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

SEARCH_BASES = [CWD.resolve(), ROOT, *ROOT.parents]


def resolve_dataset_path(path_value: str | Path) -> Path:
    raw = Path(str(path_value))
    candidates: list[Path] = []

    if raw.is_absolute():
        candidates.append(raw)
    else:
        candidates.append(raw)
        candidates.extend(base / raw for base in SEARCH_BASES)

    parts = raw.parts
    if "Scoliosis_Dataset" in parts:
            idx = parts.index("Scoliosis_Dataset")
            trimmed = Path(*parts[idx + 1 :])
            if str(trimmed) not in {"", "."}:
                candidates.append(ROOT / trimmed)

    seen: set[str] = set()
    unique_candidates: list[Path] = []
    for candidate in candidates:
        key = str(candidate)
        if key not in seen:
            seen.add(key)
            unique_candidates.append(candidate)

    for candidate in unique_candidates:
        if candidate.exists():
            return candidate.resolve()

    return unique_candidates[-1].resolve()


INDEX_PATH = ROOT / "indice_dataset.csv"
DICT_PATH = ROOT / "diccionario_etiquetas_T1_T12_L1_L5.json"
MANIFEST_PATH = PROJECT_DIR / "reports" / "analysis_outputs" / "training_manifest_thoracolumbar.csv"
BINARY_GROUP_MAP_PATH = PROJECT_DIR / "reports" / "analysis_outputs" / "training_runs_binary_thoracolumbar" / "binary_spine_group_partition_map.csv"
BINARY_MODEL_PATH = PROJECT_DIR / "models" / "binary_spine_thoracolumbar_best.pt"
MULTICLASS_MODEL_PATH = PROJECT_DIR / "models" / "thoracolumbar_partial_cascade_explained_best.pt"
BINARY_THRESHOLD_CONFIG_PATH = PROJECT_DIR / "reports" / "analysis_outputs" / "training_runs_binary_thoracolumbar" / "binary_spine_threshold_config.json"
OUTPUT_DIR = PROJECT_DIR / "reports" / "analysis_outputs" / "thoracolumbar_postprocess_anatomical_v2_explained"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for path in [INDEX_PATH, DICT_PATH, MANIFEST_PATH, BINARY_GROUP_MAP_PATH, BINARY_MODEL_PATH, MULTICLASS_MODEL_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"No existe archivo requerido: {path}")

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")
IMG_SIZE_BINARY = (512, 256)
IMG_SIZE_MULTICLASS = (640, 320)
IGNORE_INDEX = 255
BINARY_THRESHOLD = 0.50
ROI_PAD_X = 28
ROI_PAD_Y = 44
MIN_FOREGROUND_PIXELS = 24
TARGET_SUBSET = "partial"

MIN_COMPONENT_PIXELS = 48
SECONDARY_COMPONENT_RATIO = 0.35
MAX_LABEL_GAP_FOR_MAIN_BAND = 2
ALLOW_MARGIN_LABELS = 1
MIN_AREA_RATIO_TO_KEEP_EDGE_OUTLIER = 0.45
MONOTONIC_TOLERANCE_PX = 6.0
RESTORE_EDGE_MIN_AREA_RATIO = 0.30
RESTORE_NEIGHBOR_DISTANCE = 1
USE_MULTICLASS_TTA = True
N_VIS_SAMPLES = 6

if BINARY_THRESHOLD_CONFIG_PATH.exists():
    binary_threshold_config = json.loads(BINARY_THRESHOLD_CONFIG_PATH.read_text(encoding="utf-8"))
    BINARY_THRESHOLD = float(binary_threshold_config.get("selected_threshold", BINARY_THRESHOLD))

print("CWD:", CWD)
print("ROOT:", ROOT)
print("DEVICE:", DEVICE)
print("OUTPUT_DIR:", OUTPUT_DIR)


## 2. Carga de metadata y seleccion del conjunto de test

Mantenemos el mismo subconjunto `partial` y la particion `test` para que la
comparacion metodologica con notebooks previos siga siendo valida.

In [ ]:
index_df_raw = pd.read_csv(INDEX_PATH)
manifest_df = pd.read_csv(MANIFEST_PATH)
group_map_df = pd.read_csv(BINARY_GROUP_MAP_PATH)
with open(DICT_PATH, 'r', encoding='utf-8') as f:
    labels_dict = json.load(f)

index_col_map = {
    'grupo': 'split',
    'imagen': 'image',
    'id_paciente': 'patient_id',
    'ruta_radiografia': 'radiograph_path',
    'ruta_mascara_binaria': 'label_binary_path',
    'ruta_mascara_multiclase_id_png': 'multiclass_id_png',
}
index_df = index_df_raw.rename(columns=index_col_map).copy()

final_multiclass_map = {int(k): v for k, v in labels_dict['mascara_multiclase_id_png'].items()}
class_names = [final_multiclass_map[i] for i in range(len(final_multiclass_map))]
num_classes = len(class_names)
valid_multiclass_ids = set(range(num_classes))
canonical_labels = [f'T{i}' for i in range(1, 13)] + [f'L{i}' for i in range(1, 6)]
label_to_class_id = {label: idx for idx, label in enumerate(class_names)}
class_id_to_label_index = {label_to_class_id[label]: idx for idx, label in enumerate(canonical_labels)}

join_cols = ['split', 'image', 'patient_id', 'radiograph_path']
dataset_subset = index_df[join_cols + ['label_binary_path', 'multiclass_id_png']].copy()
table = manifest_df.merge(dataset_subset, on=join_cols, how='left', suffixes=('', '_idx'))
table['multiclass_mask_path'] = table['mask_path'].fillna(table['multiclass_id_png'])
table['radiograph_path_abs'] = table['radiograph_path'].apply(lambda rel: str(resolve_dataset_path(rel)))
table['binary_mask_path_abs'] = table['label_binary_path'].apply(lambda rel: str(resolve_dataset_path(rel)))
table['multiclass_mask_path_abs'] = table['multiclass_mask_path'].apply(lambda rel: str(resolve_dataset_path(rel)))

for col in ['usable_for_thoracolumbar_core', 'usable_for_thoracolumbar_partial', 'needs_annotation_review']:
    if col in table.columns:
        table[col] = table[col].map(
            lambda x: x if isinstance(x, bool) else str(x).strip().lower() == 'true'
        )

group_partition_map = group_map_df.drop_duplicates().set_index('group_id_for_split')['partition'].to_dict()
table['partition'] = table['group_id_for_split'].map(group_partition_map)

subset_flag = 'usable_for_thoracolumbar_core' if TARGET_SUBSET == 'core' else 'usable_for_thoracolumbar_partial'
analysis_df = table.loc[
    table[subset_flag] & ~table['needs_annotation_review'] & table['partition'].eq('test')
].copy().reset_index(drop=True)

print('Subset analizado:', TARGET_SUBSET)
print('Muestras de test:', len(analysis_df))
display(analysis_df.groupby('split').size().rename('images').reset_index())

## 3. Utilidades de imagen, ROI y modelos

Esta seccion reconstruye el flujo de inferencia: lectura, normalizacion, ROI binaria
y modelo multiclase.

In [ ]:
def read_gray(path: str) -> np.ndarray:
    return np.array(Image.open(path).convert('L'))


def resize_image(arr: np.ndarray, size: tuple[int, int]) -> np.ndarray:
    return np.array(Image.fromarray(arr).resize((size[1], size[0]), resample=Image.BILINEAR))


def resize_mask(arr: np.ndarray, size: tuple[int, int]) -> np.ndarray:
    return np.array(Image.fromarray(arr.astype(np.uint8)).resize((size[1], size[0]), resample=Image.NEAREST))


def build_multiclass_mask(path: str, size: tuple[int, int] | None = None) -> np.ndarray:
    raw = np.array(Image.open(path), dtype=np.int32)
    out = np.zeros_like(raw, dtype=np.uint8)
    valid_mask = np.isin(raw, list(valid_multiclass_ids))
    out[~valid_mask] = IGNORE_INDEX
    out[valid_mask] = raw[valid_mask].astype(np.uint8)
    if size is not None:
        out = resize_mask(out, size)
    return out


def bbox_from_mask(mask: np.ndarray, min_foreground_pixels: int = 24) -> tuple[int, int, int, int] | None:
    ys, xs = np.where(mask > 0)
    if len(xs) < min_foreground_pixels:
        return None
    return int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1


def clamp_bbox(bbox: tuple[int, int, int, int], image_shape: tuple[int, int]) -> tuple[int, int, int, int]:
    h, w = image_shape
    x0, y0, x1, y1 = bbox
    x0 = max(0, min(x0, w - 1))
    y0 = max(0, min(y0, h - 1))
    x1 = max(x0 + 1, min(x1, w))
    y1 = max(y0 + 1, min(y1, h))
    return x0, y0, x1, y1


def expand_bbox(
    bbox: tuple[int, int, int, int],
    image_shape: tuple[int, int],
    pad_x: int = 28,
    pad_y: int = 44,
) -> tuple[int, int, int, int]:
    x0, y0, x1, y1 = bbox
    return clamp_bbox((x0 - pad_x, y0 - pad_y, x1 + pad_x, y1 + pad_y), image_shape)


def scale_bbox(
    bbox: tuple[int, int, int, int],
    src_shape: tuple[int, int],
    dst_shape: tuple[int, int],
) -> tuple[int, int, int, int]:
    src_h, src_w = src_shape
    dst_h, dst_w = dst_shape
    x0, y0, x1, y1 = bbox
    sx = dst_w / src_w
    sy = dst_h / src_h
    scaled = (int(round(x0 * sx)), int(round(y0 * sy)), int(round(x1 * sx)), int(round(y1 * sy)))
    return clamp_bbox(scaled, dst_shape)


def crop_array(arr: np.ndarray, bbox: tuple[int, int, int, int]) -> np.ndarray:
    x0, y0, x1, y1 = bbox
    return arr[y0:y1, x0:x1]


def normalize_image(image_2d: np.ndarray) -> np.ndarray:
    mean = float(image_2d.mean())
    std = float(image_2d.std())
    if std < 1e-6:
        return image_2d - mean
    return (image_2d - mean) / std


def build_coordinate_channels(height: int, width: int) -> np.ndarray:
    y_coords = np.linspace(0.0, 1.0, height, dtype=np.float32)[:, None]
    x_coords = np.linspace(0.0, 1.0, width, dtype=np.float32)[None, :]
    y_map = np.repeat(y_coords, width, axis=1)
    x_map = np.repeat(x_coords, height, axis=0)
    return np.stack([y_map, x_map], axis=0)


class DoubleConvBinary(nn.Module):
    def __init__(self, c_in: int, c_out: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(c_in, c_out, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(c_out),
            nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(c_out),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class BinaryUNetSmall(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, base: int = 32):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.e1 = DoubleConvBinary(in_channels, base)
        self.e2 = DoubleConvBinary(base, base * 2)
        self.e3 = DoubleConvBinary(base * 2, base * 4)
        self.e4 = DoubleConvBinary(base * 4, base * 8)
        self.b = DoubleConvBinary(base * 8, base * 16)
        self.u4 = nn.ConvTranspose2d(base * 16, base * 8, kernel_size=2, stride=2)
        self.d4 = DoubleConvBinary(base * 16, base * 8)
        self.u3 = nn.ConvTranspose2d(base * 8, base * 4, kernel_size=2, stride=2)
        self.d3 = DoubleConvBinary(base * 8, base * 4)
        self.u2 = nn.ConvTranspose2d(base * 4, base * 2, kernel_size=2, stride=2)
        self.d2 = DoubleConvBinary(base * 4, base * 2)
        self.u1 = nn.ConvTranspose2d(base * 2, base, kernel_size=2, stride=2)
        self.d1 = DoubleConvBinary(base * 2, base)
        self.head = nn.Conv2d(base, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        b = self.b(self.pool(e4))
        d4 = self.d4(torch.cat([self.u4(b), e4], dim=1))
        d3 = self.d3(torch.cat([self.u3(d4), e3], dim=1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], dim=1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], dim=1))
        return self.head(d1)


class DoubleConv(nn.Module):
    def __init__(self, c_in: int, c_out: int, dropout: float = 0.0):
        super().__init__()
        layers = [
            nn.Conv2d(c_in, c_out, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(c_out),
            nn.ReLU(inplace=True),
            nn.Conv2d(c_out, c_out, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(c_out),
            nn.ReLU(inplace=True),
        ]
        if dropout > 0:
            layers.append(nn.Dropout2d(dropout))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class UNetEnhanced(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, base: int = 48, dropout: float = 0.10):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.e1 = DoubleConv(in_channels, base, dropout=0.0)
        self.e2 = DoubleConv(base, base * 2, dropout=0.0)
        self.e3 = DoubleConv(base * 2, base * 4, dropout=0.0)
        self.e4 = DoubleConv(base * 4, base * 8, dropout=dropout * 0.5)
        self.b = DoubleConv(base * 8, base * 16, dropout=dropout)
        self.u4 = nn.ConvTranspose2d(base * 16, base * 8, kernel_size=2, stride=2)
        self.d4 = DoubleConv(base * 16, base * 8, dropout=dropout * 0.5)
        self.u3 = nn.ConvTranspose2d(base * 8, base * 4, kernel_size=2, stride=2)
        self.d3 = DoubleConv(base * 8, base * 4, dropout=0.0)
        self.u2 = nn.ConvTranspose2d(base * 4, base * 2, kernel_size=2, stride=2)
        self.d2 = DoubleConv(base * 4, base * 2, dropout=0.0)
        self.u1 = nn.ConvTranspose2d(base * 2, base, kernel_size=2, stride=2)
        self.d1 = DoubleConv(base * 2, base, dropout=0.0)
        self.head = nn.Conv2d(base, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        b = self.b(self.pool(e4))
        d4 = self.d4(torch.cat([self.u4(b), e4], dim=1))
        d3 = self.d3(torch.cat([self.u3(d4), e3], dim=1))
        d2 = self.d2(torch.cat([self.u2(d3), e2], dim=1))
        d1 = self.d1(torch.cat([self.u1(d2), e1], dim=1))
        return self.head(d1)

## 4. Inferencia base

Esta celda genera la prediccion cruda del modelo sobre todas las muestras del test.
Esa prediccion cruda es el punto de partida para el postproceso v2.

In [ ]:
binary_model = BinaryUNetSmall(in_channels=1, out_channels=1).to(DEVICE)
binary_model.load_state_dict(torch.load(BINARY_MODEL_PATH, map_location=DEVICE))
binary_model.eval()

multiclass_model = UNetEnhanced(in_channels=3, out_channels=num_classes, base=48, dropout=0.10).to(DEVICE)
multiclass_model.load_state_dict(torch.load(MULTICLASS_MODEL_PATH, map_location=DEVICE))
multiclass_model.eval()


def predict_binary_roi_for_row(row: pd.Series) -> dict:
    image_raw = read_gray(row["radiograph_path_abs"])
    original_shape = image_raw.shape
    image_resized = resize_image(image_raw, IMG_SIZE_BINARY).astype(np.float32) / 255.0
    image_tensor = torch.tensor(image_resized[None, None, ...], dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        logits = binary_model(image_tensor)
        pred_mask_small = (torch.sigmoid(logits)[0, 0].detach().cpu().numpy() >= BINARY_THRESHOLD).astype(np.uint8)
    bbox_small = bbox_from_mask(pred_mask_small, min_foreground_pixels=MIN_FOREGROUND_PIXELS)
    if bbox_small is None:
        h, w = original_shape
        return {
            "bbox_x0": 0,
            "bbox_y0": 0,
            "bbox_x1": w,
            "bbox_y1": h,
            "bbox_width": w,
            "bbox_height": h,
            "roi_source": "pred_binary_fallback_full_image",
        }
    bbox_original = scale_bbox(bbox_small, src_shape=IMG_SIZE_BINARY, dst_shape=original_shape)
    x0, y0, x1, y1 = expand_bbox(bbox_original, image_shape=original_shape, pad_x=ROI_PAD_X, pad_y=ROI_PAD_Y)
    return {
        "bbox_x0": x0,
        "bbox_y0": y0,
        "bbox_x1": x1,
        "bbox_y1": y1,
        "bbox_width": x1 - x0,
        "bbox_height": y1 - y0,
        "roi_source": "pred_binary",
    }


def prepare_inference_sample(row: pd.Series, roi_meta: dict) -> dict:
    image_raw = read_gray(row["radiograph_path_abs"])
    target_raw = build_multiclass_mask(row["multiclass_mask_path_abs"], size=None)
    bbox = (
        int(roi_meta["bbox_x0"]),
        int(roi_meta["bbox_y0"]),
        int(roi_meta["bbox_x1"]),
        int(roi_meta["bbox_y1"]),
    )
    image_crop = crop_array(image_raw, bbox)
    target_crop = crop_array(target_raw, bbox)
    image_crop = resize_image(image_crop, IMG_SIZE_MULTICLASS).astype(np.float32) / 255.0
    image_crop = normalize_image(image_crop)
    target_crop = resize_mask(target_crop, IMG_SIZE_MULTICLASS).astype(np.int64)
    coord_channels = build_coordinate_channels(IMG_SIZE_MULTICLASS[0], IMG_SIZE_MULTICLASS[1])
    image_channels = np.concatenate([np.expand_dims(image_crop, axis=0), coord_channels], axis=0)
    return {
        "image_channels": image_channels,
        "target_mask": target_crop,
        "image_gray": image_crop,
        "bbox": bbox,
    }


def predict_multiclass_mask_with_tta(image_channels: np.ndarray) -> np.ndarray:
    image_tensor = torch.tensor(image_channels[None, ...], dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        logits = multiclass_model(image_tensor)
        probs = torch.softmax(logits, dim=1)
        if USE_MULTICLASS_TTA:
            flipped_input = torch.flip(image_tensor, dims=[3])
            flipped_logits = multiclass_model(flipped_input)
            flipped_probs = torch.softmax(flipped_logits, dim=1)
            flipped_probs = torch.flip(flipped_probs, dims=[3])
            probs = 0.5 * (probs + flipped_probs)
        pred_mask = torch.argmax(probs, dim=1)[0].detach().cpu().numpy().astype(np.int64)
    return pred_mask


raw_pred_lookup = {}
target_lookup = {}
image_lookup = {}
roi_lookup = {}

for _, row in analysis_df.iterrows():
    roi_meta = predict_binary_roi_for_row(row)
    prepared = prepare_inference_sample(row, roi_meta)
    pred_mask = predict_multiclass_mask_with_tta(prepared["image_channels"])

    sample_id = row["unique_sample_id"]
    raw_pred_lookup[sample_id] = pred_mask
    target_lookup[sample_id] = prepared["target_mask"]
    image_lookup[sample_id] = prepared["image_gray"]
    roi_lookup[sample_id] = roi_meta

print("Inferencia base completada para", len(raw_pred_lookup), "muestras.")


## 5. Postproceso anatomico v2

Aqui cambiamos la filosofia del postproceso.

En vez de buscar una sola secuencia fuerte y borrar lo demas, hacemos:

1. limpieza de componentes pequenos
2. calculo de centroides y areas por clase
3. seleccion de una banda principal de etiquetas presentes
4. recorte suave de outliers en extremos
5. correccion conservadora de violaciones de monotonia vertical

La regla central es: **preferimos preservar antes que borrar**.

In [ ]:
def keep_reliable_components_per_class(mask_2d: np.ndarray) -> np.ndarray:
    cleaned = np.zeros_like(mask_2d, dtype=np.int64)
    for class_id in range(1, num_classes):
        class_mask = (mask_2d == class_id).astype(np.uint8)
        labeled, num_components = ndi.label(class_mask)
        if num_components == 0:
            continue

        component_ids = np.arange(1, num_components + 1)
        areas = np.array(ndi.sum(class_mask, labeled, index=component_ids), dtype=np.float64)
        largest_area = float(areas.max())
        keep_ids = []
        for comp_id, area in zip(component_ids, areas):
            if area < MIN_COMPONENT_PIXELS:
                continue
            if area >= largest_area * SECONDARY_COMPONENT_RATIO:
                keep_ids.append(int(comp_id))

        if not keep_ids and largest_area >= MIN_COMPONENT_PIXELS:
            keep_ids = [int(component_ids[np.argmax(areas)])]

        for comp_id in keep_ids:
            cleaned[labeled == comp_id] = class_id
    return cleaned


def class_stats_from_mask(mask_2d: np.ndarray) -> pd.DataFrame:
    rows = []
    total_fg = float((mask_2d > 0).sum()) + 1e-6
    for class_id in range(1, num_classes):
        class_mask = mask_2d == class_id
        if not class_mask.any():
            continue
        ys, xs = np.where(class_mask)
        rows.append({
            "class_id": class_id,
            "class_name": class_names[class_id],
            "label_index": class_id_to_label_index[class_id],
            "area_pixels": int(class_mask.sum()),
            "area_ratio": float(class_mask.sum() / total_fg),
            "centroid_y": float(np.mean(ys)),
            "centroid_x": float(np.mean(xs)),
            "y_min": int(ys.min()),
            "y_max": int(ys.max()),
        })
    if not rows:
        return pd.DataFrame(columns=[
            "class_id",
            "class_name",
            "label_index",
            "area_pixels",
            "area_ratio",
            "centroid_y",
            "centroid_x",
            "y_min",
            "y_max",
        ])
    return pd.DataFrame(rows).sort_values("label_index").reset_index(drop=True)


def build_label_blocks(label_indices: list[int], gap_tolerance: int = 2) -> list[list[int]]:
    if not label_indices:
        return []
    blocks = [[label_indices[0]]]
    for idx in label_indices[1:]:
        if idx - blocks[-1][-1] <= gap_tolerance + 1:
            blocks[-1].append(idx)
        else:
            blocks.append([idx])
    return blocks


def select_dominant_label_band(stats_df: pd.DataFrame) -> tuple[int | None, int | None]:
    if stats_df.empty:
        return None, None

    indices = stats_df["label_index"].tolist()
    blocks = build_label_blocks(indices, gap_tolerance=MAX_LABEL_GAP_FOR_MAIN_BAND)
    scored = []
    for block in blocks:
        block_df = stats_df.loc[stats_df["label_index"].isin(block)]
        score = (
            len(block),
            float(block_df["area_pixels"].sum()),
            -float(block_df["label_index"].min()),
        )
        scored.append((score, block[0], block[-1]))
    scored = sorted(scored, reverse=True)
    return int(scored[0][1]), int(scored[0][2])


def zero_out_class(mask_2d: np.ndarray, class_id: int) -> np.ndarray:
    out = mask_2d.copy()
    out[out == class_id] = 0
    return out


def trim_edge_outliers(mask_2d: np.ndarray, stats_df: pd.DataFrame, start_idx: int | None, end_idx: int | None) -> tuple[np.ndarray, list[str]]:
    if stats_df.empty or start_idx is None or end_idx is None:
        return mask_2d, []

    out = mask_2d.copy()
    decisions = []
    median_area = float(stats_df["area_pixels"].median()) if not stats_df.empty else 0.0
    allowed_start = max(0, start_idx - ALLOW_MARGIN_LABELS)
    allowed_end = min(len(canonical_labels) - 1, end_idx + ALLOW_MARGIN_LABELS)

    for _, row in stats_df.iterrows():
        label_index = int(row["label_index"])
        class_id = int(row["class_id"])
        area_pixels = float(row["area_pixels"])
        if allowed_start <= label_index <= allowed_end:
            continue
        if median_area <= 0:
            continue
        if area_pixels < median_area * MIN_AREA_RATIO_TO_KEEP_EDGE_OUTLIER:
            out = zero_out_class(out, class_id)
            decisions.append(f"edge_trim:{row['class_name']}")
    return out, decisions


def enforce_monotonic_centroids(mask_2d: np.ndarray) -> tuple[np.ndarray, list[str]]:
    out = mask_2d.copy()
    decisions = []

    while True:
        stats_df = class_stats_from_mask(out)
        if len(stats_df) <= 1:
            break

        violation_found = False
        rows = stats_df.to_dict(orient="records")
        for prev_row, curr_row in zip(rows[:-1], rows[1:]):
            prev_y = float(prev_row["centroid_y"])
            curr_y = float(curr_row["centroid_y"])
            if curr_y + MONOTONIC_TOLERANCE_PX < prev_y:
                prev_area = float(prev_row["area_pixels"])
                curr_area = float(curr_row["area_pixels"])
                drop_class = int(prev_row["class_id"]) if prev_area < curr_area else int(curr_row["class_id"])
                drop_name = prev_row["class_name"] if prev_area < curr_area else curr_row["class_name"]
                out = zero_out_class(out, drop_class)
                decisions.append(f"monotonic_drop:{drop_name}")
                violation_found = True
                break

        if not violation_found:
            break

    return out, decisions


def restore_supported_edge_labels(
    mask_2d: np.ndarray,
    reference_mask: np.ndarray,
    start_idx: int | None,
    end_idx: int | None,
) -> tuple[np.ndarray, list[str]]:
    if start_idx is None or end_idx is None:
        return mask_2d, []

    out = mask_2d.copy()
    ref_stats = class_stats_from_mask(reference_mask)
    final_stats = class_stats_from_mask(out)
    if ref_stats.empty:
        return out, []

    decisions = []
    median_area = float(ref_stats["area_pixels"].median()) if not ref_stats.empty else 0.0
    kept_indices = set(final_stats["label_index"].tolist())
    boundary_candidates = {max(0, start_idx - 1), min(len(canonical_labels) - 1, end_idx + 1)}

    for _, row in ref_stats.iterrows():
        label_index = int(row["label_index"])
        if label_index in kept_indices:
            continue
        if label_index not in boundary_candidates:
            continue
        if abs(label_index - start_idx) > RESTORE_NEIGHBOR_DISTANCE and abs(label_index - end_idx) > RESTORE_NEIGHBOR_DISTANCE:
            continue
        if median_area <= 0:
            continue
        if float(row["area_pixels"]) < median_area * RESTORE_EDGE_MIN_AREA_RATIO:
            continue
        class_id = int(row["class_id"])
        out[reference_mask == class_id] = class_id
        decisions.append(f"edge_restore:{row['class_name']}")

    return out, decisions


def postprocess_mask_v2(mask_2d: np.ndarray) -> tuple[np.ndarray, pd.DataFrame, dict]:
    cleaned = keep_reliable_components_per_class(mask_2d)
    stats_initial = class_stats_from_mask(cleaned)
    start_idx, end_idx = select_dominant_label_band(stats_initial)

    trimmed, trim_decisions = trim_edge_outliers(cleaned, stats_initial, start_idx, end_idx)
    monotonic, monotonic_decisions = enforce_monotonic_centroids(trimmed)
    restored, restore_decisions = restore_supported_edge_labels(monotonic, cleaned, start_idx, end_idx)
    stats_final = class_stats_from_mask(restored)

    meta = {
        "selected_start_label": canonical_labels[start_idx] if start_idx is not None else "",
        "selected_end_label": canonical_labels[end_idx] if end_idx is not None else "",
        "num_labels_before": int(len(class_stats_from_mask(mask_2d))),
        "num_labels_after_component_cleaning": int(len(stats_initial)),
        "num_labels_after_final": int(len(stats_final)),
        "num_positive_pixels_before": int((mask_2d > 0).sum()),
        "num_positive_pixels_after_final": int((restored > 0).sum()),
        "trim_decisions": "; ".join(trim_decisions),
        "monotonic_decisions": "; ".join(monotonic_decisions),
        "restore_decisions": "; ".join(restore_decisions),
    }
    return restored, stats_final, meta


## 6. Aplicacion del postproceso y metricas

Evaluamos dos escenarios:

- prediccion cruda
- prediccion con postproceso v2

Y medimos impacto en accuracy, Dice e IoU por regiones.

In [ ]:
def metrics_from_prediction_lookup(pred_lookup: dict[str, np.ndarray], target_lookup_in: dict[str, np.ndarray]) -> tuple[pd.DataFrame, pd.DataFrame]:
    intersection = np.zeros(num_classes, dtype=np.float64)
    pred_area = np.zeros(num_classes, dtype=np.float64)
    target_area = np.zeros(num_classes, dtype=np.float64)
    total_valid_correct = 0.0
    total_valid_pixels = 0.0

    for sample_id, pred_mask in pred_lookup.items():
        target_mask = target_lookup_in[sample_id]
        valid = target_mask != IGNORE_INDEX
        total_valid_correct += float((pred_mask[valid] == target_mask[valid]).sum())
        total_valid_pixels += float(valid.sum())
        for class_id in range(num_classes):
            pred_class = pred_mask[valid] == class_id
            target_class = target_mask[valid] == class_id
            intersection[class_id] += np.logical_and(pred_class, target_class).sum()
            pred_area[class_id] += pred_class.sum()
            target_area[class_id] += target_class.sum()

    dice = (2.0 * intersection + 1e-6) / (pred_area + target_area + 1e-6)
    iou = (intersection + 1e-6) / (pred_area + target_area - intersection + 1e-6)
    per_class_df = pd.DataFrame({
        'class_id': np.arange(num_classes),
        'class_name': class_names,
        'pred_pixels': pred_area,
        'target_pixels': target_area,
        'dice': dice,
        'iou': iou,
    })
    per_class_df['region'] = per_class_df['class_name'].map(
        lambda x: 'background' if x == 'background' else ('thoracic' if x.startswith('T') else 'lumbar')
    )
    fg_df = per_class_df.loc[per_class_df['class_id'] > 0].copy()
    summary_df = pd.DataFrame([
        {'metric': 'pixel_accuracy', 'value': float((total_valid_correct + 1e-6) / (total_valid_pixels + 1e-6))},
        {'metric': 'macro_dice_fg', 'value': float(fg_df['dice'].mean())},
        {'metric': 'macro_iou_fg', 'value': float(fg_df['iou'].mean())},
        {'metric': 'macro_dice_thoracic', 'value': float(fg_df.loc[fg_df['region'] == 'thoracic', 'dice'].mean())},
        {'metric': 'macro_dice_lumbar', 'value': float(fg_df.loc[fg_df['region'] == 'lumbar', 'dice'].mean())},
    ])
    return summary_df, per_class_df


def monotonic_summary(pred_lookup: dict[str, np.ndarray]) -> tuple[int, int]:
    ok_count = 0
    total_count = 0
    for sample_id, mask in pred_lookup.items():
        stats_df = class_stats_from_mask(mask)
        if len(stats_df) <= 1:
            continue
        total_count += 1
        ys = stats_df.sort_values('label_index')['centroid_y'].tolist()
        if all(curr >= prev - MONOTONIC_TOLERANCE_PX for prev, curr in zip(ys[:-1], ys[1:])):
            ok_count += 1
    return ok_count, total_count


postprocessed_lookup = {}
postprocess_stats_lookup = {}
postprocess_meta_rows = []

for sample_id, raw_mask in raw_pred_lookup.items():
    post_mask, stats_df, meta = postprocess_mask_v2(raw_mask)
    postprocessed_lookup[sample_id] = post_mask
    postprocess_stats_lookup[sample_id] = stats_df
    postprocess_meta_rows.append({'unique_sample_id': sample_id, **meta})

raw_summary_df, raw_per_class_df = metrics_from_prediction_lookup(raw_pred_lookup, target_lookup)
post_summary_df, post_per_class_df = metrics_from_prediction_lookup(postprocessed_lookup, target_lookup)

compare_df = raw_summary_df.merge(post_summary_df, on='metric', suffixes=('_raw', '_post_v2'))
compare_df['delta_abs'] = compare_df['value_post_v2'] - compare_df['value_raw']
compare_df['delta_rel_pct'] = 100.0 * compare_df['delta_abs'] / compare_df['value_raw'].replace(0, np.nan)

raw_monotonic_ok, raw_monotonic_total = monotonic_summary(raw_pred_lookup)
post_monotonic_ok, post_monotonic_total = monotonic_summary(postprocessed_lookup)

monotonic_df = pd.DataFrame([
    {'stage': 'raw', 'monotonic_ok': raw_monotonic_ok, 'evaluated_samples': raw_monotonic_total},
    {'stage': 'post_v2', 'monotonic_ok': post_monotonic_ok, 'evaluated_samples': post_monotonic_total},
])
monotonic_df['monotonic_ratio'] = monotonic_df['monotonic_ok'] / monotonic_df['evaluated_samples'].replace(0, np.nan)

postprocess_meta_df = pd.DataFrame(postprocess_meta_rows)

display(compare_df)
display(monotonic_df)
display(postprocess_meta_df.head())

## 7. Analisis por muestra: etiquetas extra y faltantes

Esta parte nos dice si el postproceso v2 realmente baja la sobreprediccion sin
disparar un aumento excesivo de vertebras faltantes.

In [ ]:
def present_labels_from_mask(mask_2d: np.ndarray) -> list[str]:
    ids = sorted([int(x) for x in np.unique(mask_2d) if int(x) > 0])
    return [class_names[i] for i in ids]


per_sample_compare_rows = []
for _, row in analysis_df.iterrows():
    sample_id = row['unique_sample_id']
    target_mask = target_lookup[sample_id]
    raw_mask = raw_pred_lookup[sample_id]
    post_mask = postprocessed_lookup[sample_id]

    gt_labels = present_labels_from_mask(np.where(target_mask == IGNORE_INDEX, 0, target_mask))
    raw_labels = present_labels_from_mask(raw_mask)
    post_labels = present_labels_from_mask(post_mask)

    per_sample_compare_rows.append({
        'unique_sample_id': sample_id,
        'split': row['split'],
        'image': row['image'],
        'gt_labels': ', '.join(gt_labels),
        'raw_labels': ', '.join(raw_labels),
        'post_v2_labels': ', '.join(post_labels),
        'raw_missing_count': len(set(gt_labels) - set(raw_labels)),
        'raw_extra_count': len(set(raw_labels) - set(gt_labels)),
        'post_missing_count': len(set(gt_labels) - set(post_labels)),
        'post_extra_count': len(set(post_labels) - set(gt_labels)),
        'raw_num_labels': len(raw_labels),
        'post_num_labels': len(post_labels),
    })

per_sample_compare_df = pd.DataFrame(per_sample_compare_rows)
per_sample_compare_df['extra_reduction'] = per_sample_compare_df['raw_extra_count'] - per_sample_compare_df['post_extra_count']
per_sample_compare_df['missing_change'] = per_sample_compare_df['post_missing_count'] - per_sample_compare_df['raw_missing_count']

presence_summary_df = pd.DataFrame([
    {'metric': 'mean_raw_extra_count', 'value': float(per_sample_compare_df['raw_extra_count'].mean())},
    {'metric': 'mean_post_extra_count', 'value': float(per_sample_compare_df['post_extra_count'].mean())},
    {'metric': 'mean_raw_missing_count', 'value': float(per_sample_compare_df['raw_missing_count'].mean())},
    {'metric': 'mean_post_missing_count', 'value': float(per_sample_compare_df['post_missing_count'].mean())},
    {'metric': 'samples_with_extra_reduction', 'value': int((per_sample_compare_df['extra_reduction'] > 0).sum())},
    {'metric': 'samples_with_missing_increase', 'value': int((per_sample_compare_df['missing_change'] > 0).sum())},
    {'metric': 'samples_with_any_post_prediction', 'value': int((per_sample_compare_df['post_num_labels'] > 0).sum())},
])

display(presence_summary_df)
display(
    per_sample_compare_df.sort_values(
        ['extra_reduction', 'missing_change', 'raw_extra_count'],
        ascending=[False, True, False],
    ).head(20)
)

## 8. Revision cualitativa antes vs despues

Visualizamos las muestras mas interesantes para entender el comportamiento real
del postproceso v2.

In [ ]:
def show_before_after(sample_id: str) -> None:
    image_gray = image_lookup[sample_id]
    raw_mask = raw_pred_lookup[sample_id]
    post_mask = postprocessed_lookup[sample_id]
    target_mask = target_lookup[sample_id].copy()
    target_mask[target_mask == IGNORE_INDEX] = 0

    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    axes[0].imshow(image_gray, cmap='gray')
    axes[0].set_title(f'ROI gris\n{sample_id}')
    axes[1].imshow(raw_mask, cmap='nipy_spectral', vmin=0, vmax=num_classes - 1)
    axes[1].set_title('Pred cruda')
    axes[2].imshow(post_mask, cmap='nipy_spectral', vmin=0, vmax=num_classes - 1)
    axes[2].set_title('Pred post v2')
    axes[3].imshow(target_mask, cmap='nipy_spectral', vmin=0, vmax=num_classes - 1)
    axes[3].set_title('GT')
    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.show()


sample_ids_to_show = per_sample_compare_df.sort_values(
    ['raw_extra_count', 'extra_reduction', 'missing_change'],
    ascending=[False, False, True],
)['unique_sample_id'].head(N_VIS_SAMPLES).tolist()

print('Muestras seleccionadas:', sample_ids_to_show)
for sample_id in sample_ids_to_show:
    show_before_after(sample_id)

## 9. Exportacion de resultados

Guardamos las tablas principales para integrarlas luego en la bitacora y en el
documento final de resultados y decisiones.

In [ ]:
raw_summary_path = OUTPUT_DIR / 'postprocess_v2_raw_summary.csv'
post_summary_path = OUTPUT_DIR / 'postprocess_v2_post_summary.csv'
compare_path = OUTPUT_DIR / 'postprocess_v2_metric_comparison.csv'
raw_per_class_path = OUTPUT_DIR / 'postprocess_v2_raw_per_class.csv'
post_per_class_path = OUTPUT_DIR / 'postprocess_v2_post_per_class.csv'
per_sample_path = OUTPUT_DIR / 'postprocess_v2_per_sample_compare.csv'
meta_path = OUTPUT_DIR / 'postprocess_v2_meta.csv'
monotonic_path = OUTPUT_DIR / 'postprocess_v2_monotonic_summary.csv'
presence_path = OUTPUT_DIR / 'postprocess_v2_presence_summary.csv'

raw_summary_df.to_csv(raw_summary_path, index=False)
post_summary_df.to_csv(post_summary_path, index=False)
compare_df.to_csv(compare_path, index=False)
raw_per_class_df.to_csv(raw_per_class_path, index=False)
post_per_class_df.to_csv(post_per_class_path, index=False)
per_sample_compare_df.to_csv(per_sample_path, index=False)
postprocess_meta_df.to_csv(meta_path, index=False)
monotonic_df.to_csv(monotonic_path, index=False)
presence_summary_df.to_csv(presence_path, index=False)

experiment_summary_df = pd.DataFrame([
    {'metric': 'target_subset', 'value': TARGET_SUBSET},
    {'metric': 'test_samples', 'value': int(len(analysis_df))},
    {'metric': 'raw_macro_dice_fg', 'value': float(raw_summary_df.loc[raw_summary_df['metric'] == 'macro_dice_fg', 'value'].iloc[0])},
    {'metric': 'post_v2_macro_dice_fg', 'value': float(post_summary_df.loc[post_summary_df['metric'] == 'macro_dice_fg', 'value'].iloc[0])},
    {'metric': 'raw_macro_dice_thoracic', 'value': float(raw_summary_df.loc[raw_summary_df['metric'] == 'macro_dice_thoracic', 'value'].iloc[0])},
    {'metric': 'post_v2_macro_dice_thoracic', 'value': float(post_summary_df.loc[post_summary_df['metric'] == 'macro_dice_thoracic', 'value'].iloc[0])},
    {'metric': 'raw_macro_dice_lumbar', 'value': float(raw_summary_df.loc[raw_summary_df['metric'] == 'macro_dice_lumbar', 'value'].iloc[0])},
    {'metric': 'post_v2_macro_dice_lumbar', 'value': float(post_summary_df.loc[post_summary_df['metric'] == 'macro_dice_lumbar', 'value'].iloc[0])},
    {'metric': 'mean_raw_extra_count', 'value': float(per_sample_compare_df['raw_extra_count'].mean())},
    {'metric': 'mean_post_extra_count', 'value': float(per_sample_compare_df['post_extra_count'].mean())},
    {'metric': 'samples_with_extra_reduction', 'value': int((per_sample_compare_df['extra_reduction'] > 0).sum())},
    {'metric': 'samples_with_missing_increase', 'value': int((per_sample_compare_df['missing_change'] > 0).sum())},
    {'metric': 'raw_monotonic_ratio', 'value': float(monotonic_df.loc[monotonic_df['stage'] == 'raw', 'monotonic_ratio'].iloc[0])},
    {'metric': 'post_v2_monotonic_ratio', 'value': float(monotonic_df.loc[monotonic_df['stage'] == 'post_v2', 'monotonic_ratio'].iloc[0])},
])
experiment_summary_path = OUTPUT_DIR / 'postprocess_v2_experiment_summary.csv'
experiment_summary_df.to_csv(experiment_summary_path, index=False)

display(experiment_summary_df)
print('Guardado:', raw_summary_path)
print('Guardado:', post_summary_path)
print('Guardado:', compare_path)
print('Guardado:', raw_per_class_path)
print('Guardado:', post_per_class_path)
print('Guardado:', per_sample_path)
print('Guardado:', meta_path)
print('Guardado:', monotonic_path)
print('Guardado:', presence_path)
print('Guardado:', experiment_summary_path)